# 01 Explore the datameet timetable
Read-only exploration of the raw files in `data/raw/` (unmodified). No cleaning is done; the string 'None' is reported, not fixed.

In [1]:
import json, pandas as pd
from pathlib import Path
RAW = Path("../data/raw")
pd.set_option("display.width", 200, "display.max_columns", 30)

sched = pd.DataFrame(json.load(open(RAW/"schedules.json")))
trains = pd.json_normalize(json.load(open(RAW/"trains.json"))["features"])
stations = pd.json_normalize(json.load(open(RAW/"stations.json"))["features"])
for f in ["schedules", "trains", "stations"]:
    print(f"{f}.json  {(RAW/(f + '.json')).stat().st_size/1e6:.1f} MB")

schedules.json  82.2 MB
trains.json  14.8 MB
stations.json  1.9 MB


## Size and columns

In [2]:
print("trains (trains.json):       ", trains["properties.number"].nunique())
print("trains (schedules.json):     ", sched["train_number"].nunique())
print("stations (stations.json):    ", stations["properties.code"].nunique())
print("stations (schedules.json):   ", sched["station_code"].nunique())
print("schedule rows (train stops): ", len(sched))
for name, df in [("schedules", sched), ("trains", trains), ("stations", stations)]:
    print(f"\n{name} columns:", list(df.columns))

trains (trains.json):        5208
trains (schedules.json):      5208
stations (stations.json):     8990
stations (schedules.json):    8539
schedule rows (train stops):  417080

schedules columns: ['arrival', 'day', 'train_name', 'station_name', 'station_code', 'id', 'train_number', 'departure']

trains columns: ['type', 'geometry.type', 'geometry.coordinates', 'properties.third_ac', 'properties.arrival', 'properties.from_station_code', 'properties.name', 'properties.zone', 'properties.chair_car', 'properties.first_class', 'properties.duration_m', 'properties.sleeper', 'properties.from_station_name', 'properties.number', 'properties.departure', 'properties.return_train', 'properties.to_station_code', 'properties.second_ac', 'properties.classes', 'properties.to_station_name', 'properties.duration_h', 'properties.type', 'properties.first_ac', 'properties.distance']

stations columns: ['type', 'geometry.type', 'geometry.coordinates', 'properties.state', 'properties.code', 'properties.name'

## Missing values
Columns hold real nulls and the literal string 'None' (origin stops have no arrival, terminal stops no departure).

In [3]:
for name, df in [("schedules", sched), ("trains", trains.drop(columns=["geometry.coordinates"], errors="ignore")), ("stations", stations.drop(columns=["geometry.coordinates"], errors="ignore"))]:
    nulls = df.isna().sum()
    nones = df.map(lambda v: isinstance(v, str) and v == "None").sum()
    out = pd.DataFrame({"null": nulls, "string None": nones})
    print(f"\n{name}"); print(out[(out > 0).any(axis=1)])


schedules
            null  string None
arrival        0        27835
day        22561            0
departure      0        27827

trains
                         null  string None
properties.arrival          0           15
properties.duration_m      15            0
properties.departure        0           15
properties.return_train   599            0
properties.classes         15            0
properties.duration_h      15            0
properties.distance        15            0

stations
                    null  string None
geometry.type        293            0
properties.state    4532            0
properties.zone     4532            0
properties.address  4532            0
geometry            8990            0


## Corridor B (Delhi - Kanpur - Prayagraj - Mughalsarai)
Working definition, for counting only (not cleaning): anchors are NDLS, DLI, TDL, ETW, CNB, ALD, MGS. This timetable predates the renaming of Allahabad and Mughal Sarai, so the old codes are used. A train is a *corridor train* if it stops at 2 or more distinct anchors. Its corridor stops are those between its first and last anchor. Route order comes from `id`, which is consecutive within a train. Trains running only on some days are counted once, because the file has no day-of-week information.

In [4]:
ANCHORS = ["NDLS", "DLI", "TDL", "ETW", "CNB", "ALD", "MGS"]
s = sched.sort_values(["train_number", "id"]).copy()
s["pos"] = s.groupby("train_number").cumcount()
a = s[s.station_code.isin(ANCHORS)]
g = a.groupby("train_number").agg(n_anchors=("station_code", "nunique"), lo=("pos", "min"), hi=("pos", "max"))
ct = g[g.n_anchors >= 2]
cs = s.merge(ct[["lo", "hi"]], left_on="train_number", right_index=True)
cs = cs[(cs.pos >= cs.lo) & (cs.pos <= cs.hi)]
per_station = cs.groupby("station_code").train_number.nunique().sort_values(ascending=False)

print("corridor trains (>=2 anchors):", len(ct))
print("corridor trains (>=4 anchors):", (g.n_anchors >= 4).sum())
print("corridor stations (any stop): ", len(per_station))
print("core stations (>=20 trains):  ", (per_station >= 20).sum())
print("core stations (>=5 trains):   ", (per_station >= 5).sum())
names = sched.drop_duplicates("station_code").set_index("station_code").station_name
print("\nAnchor junctions - trains stopping:")
print(per_station.reindex(ANCHORS).rename("trains").to_frame().join(names))

corridor trains (>=2 anchors): 237
corridor trains (>=4 anchors): 165
corridor stations (any stop):  435
core stations (>=20 trains):   125
core stations (>=5 trains):    235

Anchor junctions - trains stopping:
              trains     station_name
station_code                         
NDLS              89        NEW DELHI
DLI               38        OLD DELHI
TDL              187        TUNDLA JN
ETW              185           ETAWAH
CNB              203   KANPUR CENTRAL
ALD              167     ALLAHABAD JN
MGS              153  MUGHAL SARAI JN


## Full route of one example train
12303 (Howrah - New Delhi, via Kanpur and Allahabad).

In [5]:
ex = "12303"
r = sched[sched.train_number == ex].sort_values("id")
print(r.train_name.iloc[0], "| stops:", len(r), "| in corridor set:", ex in ct.index)
r = r.assign(on_corridor=r.station_code.isin(per_station.index))
print(r[["station_code", "station_name", "day", "arrival", "departure", "on_corridor"]].to_string(index=False))

Poorva Express | stops: 236 | in corridor set: True
station_code         station_name  day  arrival departure  on_corridor
         HWH            HOWRAH JN  1.0     None  08:10:00        False
         LLH               LILUAH  1.0 08:13:00  08:13:00        False
         BEQ                BELUR  1.0 08:14:00  08:14:00        False
         BLY                BALLY  1.0 08:16:00  08:16:00        False
         BZL            BELANAGAR  1.0 08:17:00  08:17:00        False
        DKAE              DANKUNI  1.0 08:20:00  08:20:00        False
        GBRA                GOBRA  1.0 08:22:00  08:22:00        False
         JOX           JANAI ROAD  1.0 08:24:00  08:24:00        False
        BPAE             BEGUMPUR  1.0 08:25:00  08:25:00        False
        BRPA            BARUIPARA  1.0 08:28:00  08:28:00        False
         MBE    MIRZAPUR BANKIPUR  1.0 08:30:00  08:30:00        False
        BLAE          BALARAMBATI  1.0 08:31:00  08:31:00        False
         KQU           KA